In [9]:
import pandas as pd
import numpy as np

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin

# Imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

df = pd.read_csv('/content/df_para_modelo.csv')

# -----------------------------
# Imputador personalizado: Percentil 75
# -----------------------------
class Quantile75Imputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.quantiles_ = np.nanpercentile(X, 75, axis=0)
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        X_filled = np.where(np.isnan(X), self.quantiles_, X)
        return X_filled

# -----------------------------
# Cargar tus datos
# -----------------------------
# Reemplaza esto con tu propio dataset
# df = pd.read_csv("ruta_a_tus_datos.csv")

# Asegura que la variable objetivo se llama 'trabajo_infantil'
X = df.drop(columns=['trabajo_infantil'])
y = df['trabajo_infantil']

# -----------------------------
# Identificar tipos de columnas
# -----------------------------
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# -----------------------------
# Pipelines de preprocesamiento
# -----------------------------
# Numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', Quantile75Imputer()),
    ('scaler', StandardScaler())
])

# Categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combinar preprocesamientos
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# -----------------------------
# Modelo final
# -----------------------------
model = LogisticRegression(penalty='l2', solver='liblinear', random_state=42)

# Pipeline completo con preprocesamiento, SMOTE y modelo
clf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', model)
])

# -----------------------------
# División estratificada del dataset
# -----------------------------
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.5, stratify=y_train_val, random_state=42
)

# -----------------------------
# Entrenamiento del modelo
# -----------------------------
clf_pipeline.fit(X_train, y_train)

# -----------------------------
# Evaluación
# -----------------------------
val_score = clf_pipeline.score(X_val, y_val)
print(f"Accuracy en validación: {val_score:.4f}")

test_score = clf_pipeline.score(X_test, y_test)
print(f"Accuracy en prueba: {test_score:.4f}")


Accuracy en validación: 0.9830
Accuracy en prueba: 0.9824
